# ShortForge — remove burned-in captions with a Colab GPU
Run the numbered sections **one at a time**, from top to bottom, using each ▶ button. No code editing needed.

First choose **Runtime → Change runtime type → T4 GPU (or another available GPU) → Save**.

This notebook uses ShortForge's original MI-GAN frame-by-frame method. AI guesses the hidden background; smudges or flicker are still possible. Your video is uploaded to your private Colab session. The original on your laptop is unchanged. Free GPUs are limited, and sessions can disconnect. Download your result before leaving.


In [ ]:
#@title 1. Set up and check the GPU
import sys, subprocess, shutil, hashlib, urllib.request, json
from pathlib import Path
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select Runtime → Change runtime type → T4 GPU → Save, then run this section again.")
subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], check=True)
if "onnxruntime" in sys.modules:
    raise RuntimeError("Setup already ran. Continue with section 2. To reinstall, restart the runtime first.")
print("Installing tools. This first step can take several minutes.")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rapidocr==3.9.2", "ipywidgets", "onnxruntime-gpu[cuda,cudnn]==1.23.2"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
import onnxruntime as ort
ort.preload_dlls(directory="")
if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError("GPU support is unavailable. Check that a GPU runtime is selected.")
WORK = Path("/content/shortforge_caption_removal")
WORK.mkdir(exist_ok=True)
MODEL = WORK / "migan_pipeline_v2.onnx"
MODEL_URL = "https://huggingface.co/Carve/MI-GAN-ONNX/resolve/c3c0c9e468934d62e79c329e35d82dd09ff8c444/migan_pipeline_v2.onnx"
MODEL_SHA = "6f1f3530a1a2324b19752018ce756088b07973cda8d7d890034ace5c8a48c40b"
if not MODEL.exists() or hashlib.sha256(MODEL.read_bytes()).hexdigest() != MODEL_SHA:
    urllib.request.urlretrieve(MODEL_URL, MODEL)
if hashlib.sha256(MODEL.read_bytes()).hexdigest() != MODEL_SHA:
    raise RuntimeError("Model download was incomplete. Restart the runtime and run setup again.")

# Standalone copy of the original ShortForge frame-by-frame worker.
"""Bounded-memory, offline MI-GAN frame inpainting. Invoked in a disposable process."""
import json
from pathlib import Path
import subprocess
import sys
import time

import cv2
import numpy as np

def ffmpeg_binary():
    return "ffmpeg"


class Cancelled(Exception):
    pass


def region_pixels(region, width, height):
    x = max(0, min(width - 1, round(region["x"] * width)))
    y = max(0, min(height - 1, round(region["y"] * height)))
    right = max(x + 1, min(width, round((region["x"] + region["width"]) * width)))
    bottom = max(y + 1, min(height, round((region["y"] + region["height"]) * height)))
    return x, y, right, bottom


def caption_mask(frame, region, detector=None):
    """Only erase OCR boxes inside the user-selected area, including text outlines."""
    height, width = frame.shape[:2]
    x, y, right, bottom = region_pixels(region, width, height)
    mask = np.zeros((height, width), dtype=np.uint8)
    if detector is None:
        mask[y:bottom, x:right] = 255
        return mask
    crop = frame[y:bottom, x:right]
    # RapidOCR accepts BGR arrays, whereas the video pipe and MI-GAN use RGB.
    result = detector(cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), use_cls=False, use_rec=False)
    boxes = getattr(result, "boxes", None)
    if boxes is not None:
        for box in boxes:
            polygon = np.round(np.asarray(box) + [x, y]).astype(np.int32)
            cv2.fillPoly(mask, [polygon], 255)
        padding = max(3, round(min(width, height) * .009))
        mask = cv2.dilate(mask, np.ones((padding * 2 + 1,) * 2, np.uint8))
        # Never let automatic removal escape the selected region.
        mask[:y] = 0
        mask[bottom:] = 0
        mask[:, :x] = 0
        mask[:, right:] = 0
    return mask


class Inpainter:
    def __init__(self, path):
        import onnxruntime as ort
        options = ort.SessionOptions()
        options.intra_op_num_threads = 2
        options.inter_op_num_threads = 1
        options.enable_cpu_mem_arena = False
        options.enable_mem_pattern = False
        options.log_severity_level = 3
        options.enable_profiling = True
        self.session = ort.InferenceSession(str(path), options, providers=[("CUDAExecutionProvider", {"use_tf32": "0"}), "CPUExecutionProvider"])

        if "CUDAExecutionProvider" not in self.session.get_providers():
            raise RuntimeError("GPU could not start. Stop here and check the setup output.")

    def __call__(self, frame, mask):
        """MI-GAN's pinned pipeline crops context and restores the original dimensions."""
        if not mask.any():
            return frame
        prediction = self.session.run(None, {
            "image": frame.transpose(2, 0, 1)[None],
            # MI-GAN uses black for erased pixels and white for retained pixels.
            "mask": (255 - mask)[None, None],
        })[0][0].transpose(1, 2, 0)
        output = frame.copy()
        output[mask > 0] = prediction[mask > 0]
        return output


def process_video(config, progress=lambda value: None, painter=None, detector=None):
    source, output = Path(config["source"]), Path(config["output"])
    cancel_path = Path(config["cancel"])
    def check_cancel():
        if cancel_path.exists():
            raise Cancelled("Caption removal cancelled. Your original is unchanged.")
    check_cancel()
    cv2.setNumThreads(1)
    if config["mask_mode"] == "text" and detector is None:
        from rapidocr import RapidOCR
        detector = RapidOCR(params={
            "Global.use_cls": False, "Global.use_rec": False,
            # A narrow caption band must not be enlarged until its short side
            # reaches 736px: that turns a small crop into a huge OCR image.
            "Det.limit_type": "max",
            "EngineConfig.onnxruntime.intra_op_num_threads": 2,
            "EngineConfig.onnxruntime.inter_op_num_threads": 1,
            "Global.log_level": "error",
        })
    check_cancel()
    painter = painter or Inpainter(config["model"])
    width, height = config["width"], config["height"]
    fps = config["fps"]
    expected = max(1, round(config["duration"] * fps))
    silent = output.with_name("silent.mp4")
    ffmpeg = ffmpeg_binary()
    decoder = encoder = None
    count = changed = 0
    begun = time.monotonic()
    try:
        decoder = subprocess.Popen([
            ffmpeg, "-v", "error", "-nostdin", "-threads", "2", "-ss", str(config["start"]),
            "-i", str(source), "-t", str(config["duration"]), "-an", "-sn", "-dn",
            "-vf", f"fps={fps},scale={width}:{height}", "-pix_fmt", "rgb24",
            "-threads", "2", "-f", "rawvideo", "-",
        ], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        encoder = subprocess.Popen([
            ffmpeg, "-v", "error", "-nostdin", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
            "-s", f"{width}x{height}", "-r", str(fps), "-i", "-", "-an",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p",
            "-threads", "2", "-movflags", "+faststart", str(silent),
        ], stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
        frame_bytes = width * height * 3
        while True:
            check_cancel()
            raw = decoder.stdout.read(frame_bytes)
            if not raw:
                break
            if len(raw) != frame_bytes:
                raise RuntimeError("The video stopped decoding. Try importing it again.")
            frame = np.frombuffer(raw, np.uint8).reshape(height, width, 3)
            mask = caption_mask(frame, config["region"], detector if config["mask_mode"] == "text" else None)
            if mask.any():
                frame = painter(frame, mask)
                changed += 1
            check_cancel()
            encoder.stdin.write(frame.tobytes())
            count += 1
            progress({"progress": min(94, 5 + 89 * count / expected), "frames": count,
                      "total_frames": expected,
                      "seconds_remaining": round((time.monotonic() - begun) / count * max(0, expected - count))})
        encoder.stdin.close()
        if decoder.wait(timeout=30) or encoder.wait(timeout=60):
            raise RuntimeError("The cleaned video could not be encoded. Try a shorter selection.")
        if count < max(1, expected - 2):
            raise RuntimeError("The video ended before the selected range. Reimport the source and try again.")
        if not changed:
            raise ValueError("No text was found in this area. Adjust the box or choose ‘Erase the whole area’, then preview again.")
        check_cancel()
        progress({"progress": 96, "frames": count, "total_frames": expected, "seconds_remaining": 0})
        result = subprocess.run([
            ffmpeg, "-v", "error", "-nostdin", "-y", "-ss", str(config["start"]), "-i", str(source),
            "-i", str(silent), "-map", "1:v:0", "-map", "0:a:0?", "-c:v", "copy", "-c:a", "aac",
            "-b:a", "192k", "-threads", "2", "-t", str(count / fps), "-movflags", "+faststart", str(output),
        ], capture_output=True, timeout=180)
        if result.returncode:
            raise RuntimeError("The soundtrack could not be kept with the cleaned video. Try importing the source again.")
        check_cancel()
        return {"frames": count, "frames_changed": changed, "seconds": round(time.monotonic() - begun, 1)}
    finally:
        for process in (decoder, encoder):
            if process is not None:
                if process.poll() is None:
                    process.kill()
                process.wait()
                for stream in (process.stdin, process.stdout):
                    if stream and not stream.closed:
                        stream.close()
        silent.unlink(missing_ok=True)


# Verify that convolution (the expensive AI operation) really runs on CUDA.
probe_painter = Inpainter(MODEL)
probe_frame = np.full((512, 512, 3), 127, dtype=np.uint8)
probe_mask = np.zeros((512, 512), dtype=np.uint8)
probe_mask[220:280, 80:430] = 255
probe_painter(probe_frame, probe_mask)
profile_file = Path(probe_painter.session.end_profiling())
events = json.loads(profile_file.read_text())
profile_file.unlink(missing_ok=True)
if not any(e.get("args", {}).get("provider") == "CUDAExecutionProvider" and "Conv" in e.get("args", {}).get("op_name", "") for e in events):
    raise RuntimeError("The AI did not use the GPU. Do not process the full clip; share this error for troubleshooting.")
PAINTER = probe_painter
print("GPU ready — AI processing verified. Continue with section 2.")


In [ ]:
#@title 2. Upload one video from your laptop
from google.colab import files, output as colab_output
from IPython.display import display, Video, clear_output
import ipywidgets as widgets
colab_output.enable_custom_widget_manager()
print("Choose your downloaded video file.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please run this section again and choose exactly one video.")
name, video_bytes = next(iter(uploaded.items()))
SOURCE = WORK / ("source" + (Path(name).suffix.lower() or ".mp4"))
SOURCE.write_bytes(video_bytes)
del uploaded, video_bytes
probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_streams", "-show_format", "-of", "json", str(SOURCE)], capture_output=True, text=True, check=True)
info = json.loads(probe.stdout)
stream = info["streams"][0]
num, den = stream["avg_frame_rate"].split("/")
FPS = float(num) / float(den)
DURATION = float(info["format"]["duration"])
cap = cv2.VideoCapture(str(SOURCE))
ok, first_frame = cap.read()
cap.release()
if not ok or not (0 < FPS <= 120) or not (0 < DURATION <= 600):
    raise ValueError("Use a readable video of up to 10 minutes and at most 120 frames per second.")
HEIGHT, WIDTH = first_frame.shape[:2]
WIDTH, HEIGHT = WIDTH // 2 * 2, HEIGHT // 2 * 2
if WIDTH * HEIGHT > 3840 * 2160:
    raise ValueError("Use a video at 4K resolution or below.")
PREVIEW_SIGNATURE = None
FINAL_SIGNATURE = None
print(f"Uploaded: {name} — {DURATION:.1f} seconds. Continue with section 3.")


In [ ]:
#@title 3. Mark the caption area
from PIL import Image, ImageDraw
horizontal = widgets.IntRangeSlider(value=[5,95], min=0, max=100, description="Left/right", continuous_update=False)
vertical = widgets.IntRangeSlider(value=[60,78], min=0, max=100, description="Top/bottom", continuous_update=False)
sample_time = widgets.FloatSlider(value=min(1., DURATION/2), min=0, max=max(0, DURATION-1/FPS), step=0.1, description="View second", continuous_update=False)
mode = widgets.Dropdown(options=[("Find text inside box (recommended)", "text"), ("Erase everything inside box", "area")], value="text", description="Removal")
area_display = widgets.Output()
def show_area(*_):
    cap = cv2.VideoCapture(str(SOURCE))
    cap.set(cv2.CAP_PROP_POS_MSEC, sample_time.value * 1000)
    ok, frame = cap.read()
    cap.release()
    with area_display:
        clear_output(wait=True)
        if not ok:
            print("Move View second slightly earlier.")
            return
        pic = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        pic.thumbnail((480,640))
        draw = ImageDraw.Draw(pic)
        x1,x2 = horizontal.value
        y1,y2 = vertical.value
        draw.rectangle((x1*pic.width/100,y1*pic.height/100,x2*pic.width/100,y2*pic.height/100), outline="red", width=3)
        display(pic)
for control in [horizontal,vertical,sample_time]:
    control.observe(show_area, names="value")
print("Move the sliders so the red box surrounds all captions, including their outlines. Use View second to check other moments. Keep unrelated text outside the box.")
display(widgets.VBox([horizontal,vertical,sample_time,mode,area_display]))
show_area()
def selection():
    x1,x2 = horizontal.value
    y1,y2 = vertical.value
    if x2 <= x1 or y2 <= y1:
        raise ValueError("Make the red box wider and taller.")
    return {"x":x1/100,"y":y1/100,"width":(x2-x1)/100,"height":(y2-y1)/100}
def signature():
    return (str(SOURCE), tuple(horizontal.value), tuple(vertical.value), mode.value)
def run_removal(start, duration, destination):
    config = {"source":str(SOURCE),"output":str(destination),"cancel":str(WORK / "cancel-unused"),"model":str(MODEL),"width":WIDTH,"height":HEIGHT,"fps":FPS,"start":start,"duration":duration,"region":selection(),"mask_mode":mode.value}
    bar = widgets.IntProgress(min=0,max=100,description="Processing")
    status = widgets.Label(value="Starting…")
    display(widgets.VBox([bar,status]))
    last_update = [0.]
    def update(p):
        if time.monotonic() - last_update[0] >= 1:
            bar.value = int(p["progress"])
            status.value = f'{p["frames"]}/{p["total_frames"]} frames; estimated time left: {p["seconds_remaining"]//60}m {p["seconds_remaining"]%60}s'
            last_update[0] = time.monotonic()
    try:
        stats = process_video(config, progress=update, painter=PAINTER)
    except BaseException:
        destination.unlink(missing_ok=True)
        bar.bar_style = "danger"
        status.value = "Stopped. Read the message below."
        raise
    bar.value = 100
    bar.bar_style = "success"
    status.value = f'Done in {stats["seconds"]:.0f} seconds.'
    return stats


In [ ]:
#@title 4. Test two seconds first
PREVIEW_SIGNATURE = None
preview_start = min(sample_time.value, max(0, DURATION - 2))
PREVIEW = WORK / "preview.mp4"
run_removal(preview_start, min(2, DURATION-preview_start), PREVIEW)
PREVIEW_SIGNATURE = signature()
display(Video(str(PREVIEW), embed=True, width=360))
print("Play the preview. If needed, adjust the box above and run this test again. If you like it, continue with section 5.")


In [ ]:
#@title 5. Process the full video
if PREVIEW_SIGNATURE != signature():
    raise ValueError("Run section 4 with your current selection first, and check its preview.")
FINAL_SIGNATURE = None
FINAL = WORK / "captions-removed.mp4"
run_removal(0, DURATION, FINAL)
FINAL_SIGNATURE = signature()
print("Full video ready, with audio retained. Run section 6 to download it.")


In [ ]:
#@title 6. Download the finished video
if FINAL_SIGNATURE != signature() or not FINAL.is_file():
    raise ValueError("Finish section 5 with the current selection first.")
files.download(str(FINAL))
print("Your browser should download captions-removed.mp4. You can import it into ShortForge and add your own captions.")


## Notes and model license
This notebook is standalone: no GitHub deployment or ShortForge server is required. Text detection and video encoding may use CPU; the MI-GAN neural network is verified on GPU. It processes every frame, retaining the original frame rate and using high-quality H.264 output; it does not use the discarded fast tracking method. Odd image dimensions are rounded down by one pixel for MP4 compatibility.

Free GPU access and session length are controlled by Google. There is no guaranteed completion time. If Colab has no GPU available, try again later. Download finished files before disconnecting. This notebook has been checked locally; a live Colab GPU run still needs to be verified.

[Colab usage and limits](https://research.google.com/colaboratory/faq.html) · [ONNX Runtime GPU support](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html)

### MI-GAN license
```text
MI-GAN model weights: migan_pipeline_v2.onnx
Source: https://github.com/Picsart-AI-Research/MI-GAN
Weights license: https://github.com/Picsart-AI-Research/MI-GAN/blob/main/LICENSE-WEIGHTS
ONNX: https://huggingface.co/andraniksargsyan/migan

MIT License

Copyright (c) 2024 Picsart AI Research (PAIR)

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```
